In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# ----------------------------------------------------
# 1. 全局超参数定义 (在真实架构中，这些通常写进 Config 类)
# ----------------------------------------------------
batch_size = 32    # B: 并行处理的序列数量
block_size = 8     # T: 序列的最大上下文长度 
n_embd = 32        # C: 每一个 token 被映射到的特征维度 

head_size = 16     # 单个注意力头投影后的特征维度 

torch.manual_seed(1337) # 保证每次输出一样，方便调试

# ----------------------------------------------------
# 2. 模拟底层的输入数据流动
# ----------------------------------------------------
x = torch.randn(batch_size, block_size, n_embd)

In [ ]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        
        self.key = nn.Linear(n_embd,head_size,bias=False)#Wk,Wq形状都是[n_embd,head_size]
        self.query = nn.Linear(n_embd,head_size,bias=False)
        self.value = nn.Linear(n_embd,head_size,bias=False)

        # register_buffer 用于注册一个不需要梯度更新的持久化状态（即掩码矩阵）
        # 我们必须按照模型能容纳的“最大上下文长度 (block_size)”来申请这个全 1 的下三角矩阵。
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self,x):
        B,T,C = x.shape

        #计算Q，K
        k = self.key(x) #执行的是高维的矩阵乘法k=x*Wk
        q = self.query(x)

        #计算注意力分数（包含缩放点积注意力）
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5#输出矩阵[T,T]

        #因果掩码
        # 动态切片截取：只从预先分配好的大矩阵 self.tril 中，切出当前句子实际长度的 [:T, :T] 块
        wei = wei.masked_fill(self.tril[:T,:T] == 0,float('-inf'))

        #归一化为概率分布
        wei = F.softmax(wei,dim=-1)#形状[B,T,T]

        #信息聚合
        v = self.value(x)
        out = wei @ v

        return out